# Information Retrieval Search Engine
## Document Verification, Misinformation and LLM Hallucinations

**Student:** Aitana Cantó Rico

**Course:** Information Retrieval  

**Professor:** Panagiota Tselenti

**University:** University of West Attica


This notebook presents the full implementation of a document-based search engine,
including data collection, indexing, querying, ranking, and evaluation.
The system is aligned with the topic of **document verification, misinformation,
and hallucination detection in AI-generated text**.


## 1. Introduction and Motivation

The rapid growth of AI-generated content has raised major concerns regarding
**misinformation, factuality, trustworthiness, and hallucinations in large language models (LLMs)**.

The goal of this project is to design and evaluate a **classical Information Retrieval
(IR) search engine** capable of retrieving relevant academic literature related to:

- Document and identity verification
- Fact checking and claim verification
- AI-generated misinformation
- LLM hallucination detection
- Groundedness in Retrieval-Augmented Generation (RAG)

This project follows the classical IR pipeline:
1. Data collection
2. Text preprocessing
3. Inverted index construction
4. Boolean and ranked retrieval
5. Evaluation using standard IR metrics


In [ ]:
# Basic imports
import os
import re
import json
import math
import pickle
import time
import requests
from collections import defaultdict, Counter
from typing import Dict, List, Tuple


## 2. Data Collection

Academic documents are collected from **OpenAlex** using keyword-based queries.
For each paper we retrieve:

- Title
- Abstract
- Publication year
- Venue
- URL / DOI
- Citation count

The dataset focuses on document verification, misinformation, and LLM hallucinations.


In [ ]:
import os
import time
import json
import requests
from typing import Dict, List, Optional, Set, Any

BASE_URL = "https://api.openalex.org/works"

MAILTO = os.getenv("OPENALEX_MAILTO", "aitanacantorico@hotmail.com")

KEYWORDS = [
    "document verification",
    "document validation",
    "document authenticity",
    "fact checking document text",
    "information credibility trustworthiness",
    "LLM hallucination detection",
    "AI-generated text reliability trust verification",
    "misinformation detection text document",
    "consistency checking generated text LLM",
]

def reconstruct_abstract(inverted_index: Optional[Dict[str, List[int]]]) -> str:

    if not inverted_index:
        return ""

    # building list (pos -> word)
    positions: Dict[int, str] = {}
    for word, pos_list in inverted_index.items():
        for p in pos_list:
            positions[p] = word

    if not positions:
        return ""

    max_pos = max(positions.keys())
    tokens = [positions.get(i, "") for i in range(max_pos + 1)]
    return " ".join([t for t in tokens if t])

def openalex_get(params: Dict[str, Any], timeout: int = 30) -> Dict:
    r = requests.get(BASE_URL, params=params, timeout=timeout)
    if r.status_code == 429:
        time.sleep(5)
        r = requests.get(BASE_URL, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def collect_for_keyword(
    keyword: str,
    target_new: int = 80,
    per_page: int = 200,
    sleep_s: float = 0.6,
    seen: Optional[Set[str]] = None,
    max_pages: int = 20
) -> List[Dict]:
    if seen is None:
        seen = set()

    collected: List[Dict] = []
    cursor = "*"  # cursor pagination

    for _ in range(max_pages):
        if len(collected) >= target_new:
            break

        params = {
            "search": keyword,
            "per-page": per_page,
            "cursor": cursor,
            "mailto": MAILTO
        }

        data = openalex_get(params=params)
        results = data.get("results", [])
        cursor = data.get("meta", {}).get("next_cursor")

        if not results:
            break

        for w in results:
            wid = w.get("id")  # OpenAlex work id (URL)
            if not wid or wid in seen:
                continue

            title = (w.get("title") or "").strip()
            abstract = reconstruct_abstract(w.get("abstract_inverted_index"))
            year = w.get("publication_year")
            venue = (w.get("primary_location") or {}).get("source", {}) or {}
            venue_name = venue.get("display_name")

            if not abstract and not title:
                continue

            doc = {
                "doc_id": wid,
                "title": title,
                "abstract": abstract,
                "year": year,
                "venue": venue_name,
                "url": (w.get("primary_location") or {}).get("landing_page_url") or w.get("doi"),
                "citationCount": w.get("cited_by_count", 0),
                "keywords_query": keyword
            }

            seen.add(wid)
            collected.append(doc)

            if len(collected) >= target_new:
                break

        time.sleep(sleep_s)

        if not cursor:
            break

    return collected

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def save_jsonl(path: str, docs: List[Dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for d in docs:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

if __name__ == "__main__":
    ensure_dir("data/raw")
    out_path = "data/raw/raw_papers.jsonl"

    seen_ids: Set[str] = set()
    all_docs: List[Dict] = []

    TARGET_TOTAL = 550

    for kw in KEYWORDS:
        if len(all_docs) >= TARGET_TOTAL:
            break

        print(f"\n--- Collecting for keyword: {kw} ---")
        new_docs = collect_for_keyword(
            keyword=kw,
            target_new=90,     # 80–120
            per_page=200,
            sleep_s=0.6,
            seen=seen_ids,
            max_pages=20
        )
        print(f"Keyword='{kw}' -> collected {len(new_docs)} new docs")
        all_docs.extend(new_docs)
        print(f"Running total: {len(all_docs)} docs")

    print(f"\nTOTAL collected: {len(all_docs)}")
    save_jsonl(out_path, all_docs)
    print(f"Saved to {out_path}")

# NOTE:
# This cell is usually executed once.
# The resulting file is saved as: data/raw/raw_papers.jsonl



--- Collecting for keyword: document verification ---
Keyword='document verification' -> collected 90 new docs
Running total: 90 docs

--- Collecting for keyword: document validation ---
Keyword='document validation' -> collected 90 new docs
Running total: 180 docs

--- Collecting for keyword: document authenticity ---
Keyword='document authenticity' -> collected 90 new docs
Running total: 270 docs

--- Collecting for keyword: fact checking document text ---
Keyword='fact checking document text' -> collected 90 new docs
Running total: 360 docs

--- Collecting for keyword: information credibility trustworthiness ---
Keyword='information credibility trustworthiness' -> collected 90 new docs
Running total: 450 docs

--- Collecting for keyword: LLM hallucination detection ---
Keyword='LLM hallucination detection' -> collected 90 new docs
Running total: 540 docs

--- Collecting for keyword: AI-generated text reliability trust verification ---
Keyword='AI-generated text reliability trust ve

## 3. Text Preprocessing

Before indexing, documents are normalized and tokenized:

- Lowercasing
- Removal of punctuation
- Stopword removal
- Minimum token length filtering

The final representation stores:
- Clean text
- Token list
- Document length

In [ ]:
import os
import re
import json
from typing import List, Dict

RAW_PATH = "data/raw/raw_papers.jsonl"
OUT_DIR = "data/processed"
OUT_PATH = os.path.join(OUT_DIR, "processed_papers.jsonl")

STOPWORDS = {
    "a","an","the","and","or","but","if","while","with","without","of","to","in","on","for","from",
    "by","as","at","is","are","was","were","be","been","being","this","that","these","those",
    "it","its","they","them","their","we","our","you","your","i","me","my",
    "can","could","should","would","may","might","must","will","shall",
    "not","no","nor","do","does","did","doing","done",
    "have","has","had","having",
    "also","such","than","then","there","here","when","where","why","how",
    "into","over","under","about","between","among","during","before","after",
}

TOKEN_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")  # only words

def normalize_text(s: str) -> str:
    s = s or ""
    s = s.replace("\u00ad", "")  # soft hyphen
    s = s.replace("\n", " ").strip()
    return s

def tokenize(text: str) -> List[str]:
    text = text.lower()
    tokens = TOKEN_RE.findall(text)

    clean = []
    for t in tokens:
        if t in STOPWORDS:
            continue
        if len(t) < 2:      # avoid tokens de 1 letter
            continue
        clean.append(t)
    return clean

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def save_jsonl(path: str, docs: List[Dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for d in docs:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

if __name__ == "__main__":
    ensure_dir(OUT_DIR)

    processed = []
    n_in = 0
    n_skipped = 0

    for doc in iter_jsonl(RAW_PATH):
        n_in += 1

        doc_id = doc.get("doc_id") or doc.get("paperId") or doc.get("id")
        title = normalize_text(doc.get("title", ""))
        abstract = normalize_text(doc.get("abstract", ""))

        # final text for indexing
        text = (title + "\n" + abstract).strip()

        if not doc_id or not text:
            n_skipped += 1
            continue

        tokens = tokenize(text)

        if len(tokens) < 5:  # filter docs too much empty
            n_skipped += 1
            continue

        processed.append({
            "doc_id": doc_id,
            "title": title,
            "abstract": abstract,
            "year": doc.get("year"),
            "venue": doc.get("venue"),
            "url": doc.get("url"),
            "citationCount": doc.get("citationCount", 0),
            "text": text,
            "tokens": tokens,
            "len_tokens": len(tokens),
            "keywords_query": doc.get("keywords_query")
        })

    save_jsonl(OUT_PATH, processed)

    print(f"Input docs: {n_in}")
    print(f"Processed docs: {len(processed)}")
    print(f"Skipped docs: {n_skipped}")
    print(f"Saved to: {OUT_PATH}")

# Output:
# data/processed/processed_papers.jsonl


Input docs: 630
Processed docs: 613
Skipped docs: 17
Saved to: data/processed/processed_papers.jsonl


## 4. Inverted Index Construction

An inverted index is built mapping each term to the list of documents
in which it appears along with term frequency.

Additional structures:
- Document frequency (df)
- Document lengths
- Metadata for result display


In [ ]:
import os
import json
import pickle
from collections import defaultdict, Counter
from typing import Dict, List, Tuple

IN_PATH = "data/processed/processed_papers.jsonl"
OUT_DIR = "data/index"

POSTINGS_PATH = os.path.join(OUT_DIR, "postings.pkl")      # term -> list[(doc_id, tf)]
VOCAB_PATH    = os.path.join(OUT_DIR, "vocab.pkl")         # term -> df
DOC_LEN_PATH  = os.path.join(OUT_DIR, "doc_lengths.pkl")   # doc_id -> len_tokens
DOC_META_PATH = os.path.join(OUT_DIR, "doc_meta.pkl")      # doc_id -> {title, year, venue, url}

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def save_pickle(path: str, obj) -> None:
    with open(path, "wb") as f:
        pickle.dump(obj, f)

if __name__ == "__main__":
    ensure_dir(OUT_DIR)

    postings: Dict[str, List[Tuple[str, int]]] = defaultdict(list)
    df: Dict[str, int] = defaultdict(int)
    doc_lengths: Dict[str, int] = {}
    doc_meta: Dict[str, dict] = {}

    n_docs = 0

    for doc in iter_jsonl(IN_PATH):
        doc_id = doc["doc_id"]
        tokens = doc["tokens"]
        n_docs += 1

        # doc length
        doc_lengths[doc_id] = int(doc.get("len_tokens", len(tokens)))

        # metadata
        doc_meta[doc_id] = {
            "title": doc.get("title", ""),
            "year": doc.get("year", None),
            "venue": doc.get("venue", None),
            "url": doc.get("url", None),
            "citationCount": doc.get("citationCount", 0),
        }

        # term frequencies
        tf_counts = Counter(tokens)

        # ipdate postings y df
        for term, tf in tf_counts.items():
            postings[term].append((doc_id, int(tf)))
            df[term] += 1

    # ordering by doc_id
    for term in postings:
        postings[term].sort(key=lambda x: x[0])

    # Saving
    save_pickle(POSTINGS_PATH, dict(postings))
    save_pickle(VOCAB_PATH, dict(df))
    save_pickle(DOC_LEN_PATH, doc_lengths)
    save_pickle(DOC_META_PATH, doc_meta)

    vocab_size = len(df)
    print(f"Indexed documents: {n_docs}")
    print(f"Vocabulary size (unique terms): {vocab_size}")
    print(f"Saved postings to: {POSTINGS_PATH}")
    print(f"Saved vocab(df) to: {VOCAB_PATH}")
    print(f"Saved doc lengths to: {DOC_LEN_PATH}")
    print(f"Saved doc meta to: {DOC_META_PATH}")


Indexed documents: 613
Vocabulary size (unique terms): 9714
Saved postings to: data/index/postings.pkl
Saved vocab(df) to: data/index/vocab.pkl
Saved doc lengths to: data/index/doc_lengths.pkl
Saved doc meta to: data/index/doc_meta.pkl


## 5. Boolean Retrieval

The system supports Boolean queries using:
- AND
- OR
- NOT
- Parentheses

This allows precise filtering of documents before ranking.


In [ ]:
import re
import pickle
from typing import List, Set, Dict, Tuple

POSTINGS_PATH = "data/index/postings.pkl"
DOC_META_PATH = "data/index/doc_meta.pkl"
DOC_LEN_PATH  = "data/index/doc_lengths.pkl"

# Tokenizator for queries
TOKEN_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def normalize_query_tokens(q: str) -> List[str]:
    # converting to tokens and to uppercase operators
    # keep AND/OR/NOT and brackets
    q = q.strip()

    # separating brackets
    q = q.replace("(", " ( ").replace(")", " ) ")

    parts = q.split()
    out = []
    for p in parts:
        up = p.upper()
        if up in {"AND", "OR", "NOT", "(", ")"}:
            out.append(up)
        else:
            # normal word -> lowercase token
            toks = TOKEN_RE.findall(p.lower())
            # ignoring weird toks
            out.extend(toks)
    return out

def get_docs_for_term(term: str, postings: Dict[str, List[Tuple[str, int]]]) -> Set[str]:
    plist = postings.get(term, [])
    return set(doc_id for doc_id, _tf in plist)

# ----- Parser (Shunting-yard) to convert to RPN -----
# Order: NOT > AND > OR
PRECEDENCE = {"NOT": 3, "AND": 2, "OR": 1}
RIGHT_ASSOC = {"NOT"}

def to_rpn(tokens: List[str]) -> List[str]:
    output = []
    ops = []

    for t in tokens:
        if t in {"AND", "OR", "NOT"}:
            while ops:
                top = ops[-1]
                if top == "(":
                    break
                if (PRECEDENCE[top] > PRECEDENCE[t]) or (
                    PRECEDENCE[top] == PRECEDENCE[t] and t not in RIGHT_ASSOC
                ):
                    output.append(ops.pop())
                else:
                    break
            ops.append(t)
        elif t == "(":
            ops.append(t)
        elif t == ")":
            while ops and ops[-1] != "(":
                output.append(ops.pop())
            if not ops:
                raise ValueError("Mismatched parentheses in query.")
            ops.pop()  # remove "("
        else:
            output.append(t)

    while ops:
        top = ops.pop()
        if top in {"(", ")"}:
            raise ValueError("Mismatched parentheses in query.")
        output.append(top)

    return output

def eval_rpn(rpn: List[str], postings: Dict[str, List[Tuple[str, int]]], universe: Set[str]) -> Set[str]:
    stack: List[Set[str]] = []

    for t in rpn:
        if t == "NOT":
            if not stack:
                raise ValueError("NOT operator missing operand.")
            a = stack.pop()
            stack.append(universe - a)
        elif t in {"AND", "OR"}:
            if len(stack) < 2:
                raise ValueError(f"{t} operator missing operand(s).")
            b = stack.pop()
            a = stack.pop()
            stack.append(a & b if t == "AND" else a | b)
        else:
            stack.append(get_docs_for_term(t, postings))

    if len(stack) != 1:
        raise ValueError("Invalid query (could not reduce to single result set).")
    return stack[0]

def pretty_print(doc_ids: List[str], doc_meta: Dict[str, dict], max_show: int = 10) -> None:
    print(f"\nResults: {len(doc_ids)} documents")
    for i, did in enumerate(doc_ids[:max_show], 1):
        m = doc_meta.get(did, {})
        title = m.get("title", "(no title)")
        year = m.get("year", "")
        venue = m.get("venue", "")
        url = m.get("url", "")
        print(f"{i:02d}. {title}")
        print(f"    {year} | {venue}")
        if url:
            print(f"    {url}")
    if len(doc_ids) > max_show:
        print(f"... ({len(doc_ids) - max_show} more)")

if __name__ == "__main__":
    postings = load_pickle(POSTINGS_PATH)
    doc_meta = load_pickle(DOC_META_PATH)
    doc_lengths = load_pickle(DOC_LEN_PATH)

    universe = set(doc_lengths.keys())
    print(f"Loaded index. Docs in universe: {len(universe)} | Terms: {len(postings)}")

    print("\nBoolean Search ready.")
    print("Examples:")
    print("  document AND verification")
    print("  (llm OR \"ai-generated\") AND NOT misinformation")
    print("  fact AND checking AND (trust OR credibility)")
    print("\nOperators: AND, OR, NOT, parentheses (). Type 'exit' to quit.\n")

    while True:
        q = input("query> ").strip()
        if not q:
            continue
        if q.lower() in {"exit", "quit"}:
            break

        try:
            tokens = normalize_query_tokens(q)
            rpn = to_rpn(tokens)
            result_set = eval_rpn(rpn, postings, universe)

            # Ordering by year desc, luego citationCount desc
            result_list = list(result_set)
            result_list.sort(
                key=lambda did: (
                    doc_meta.get(did, {}).get("year") or 0,
                    doc_meta.get(did, {}).get("citationCount") or 0
                ),
                reverse=True
            )

            pretty_print(result_list, doc_meta, max_show=10)

        except Exception as e:
            print(f"Error: {e}")


Loaded index. Docs in universe: 613 | Terms: 9714

Boolean Search ready.
Examples:
  document AND verification
  (llm OR "ai-generated") AND NOT misinformation
  fact AND checking AND (trust OR credibility)

Operators: AND, OR, NOT, parentheses (). Type 'exit' to quit.

query> document AND verification

Results: 90 documents
01. Optimizing Cryptographic and Image hashing for document verification
    2025 | SSRN Electronic Journal
    https://doi.org/10.2139/ssrn.5112131
02. Blockchain-Based Decentralized Document Verification and Its Applications
    2025 | Journal of Information Systems Engineering & Management
    https://doi.org/10.52783/jisem.v10i10s.1362
03. Digital Document Verification using Blockchain
    2025 | International Journal for Research in Applied Science and Engineering Technology
    https://doi.org/10.22214/ijraset.2025.68159
04. BLOCKCHAIN-BASED DOCUMENT VERIFICATION: A COMPREHENSIVE REVIEW
    2025 | International Journal of Advanced Research in Computer Science

## 6. Ranked Retrieval using TF-IDF

Documents are ranked using the TF-IDF weighting scheme:

- Term Frequency: log-scaled
- Inverse Document Frequency: smoothed IDF
- Document score = sum(TF × IDF)

This allows ranking documents by estimated relevance.


In [ ]:
import re
import math
import pickle
from collections import Counter, defaultdict
from typing import Dict, List, Tuple

POSTINGS_PATH = "data/index/postings.pkl"     # term -> list[(doc_id, tf)]
VOCAB_PATH    = "data/index/vocab.pkl"        # term -> df
DOC_LEN_PATH  = "data/index/doc_lengths.pkl"  # doc_id -> len_tokens
DOC_META_PATH = "data/index/doc_meta.pkl"     # doc_id -> goal

TOKEN_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

# Same stopwords as preprocessing
STOPWORDS = {
    "a","an","the","and","or","but","if","while","with","without","of","to","in","on","for","from",
    "by","as","at","is","are","was","were","be","been","being","this","that","these","those",
    "it","its","they","them","their","we","our","you","your","i","me","my",
    "can","could","should","would","may","might","must","will","shall",
    "not","no","nor","do","does","did","doing","done",
    "have","has","had","having",
    "also","such","than","then","there","here","when","where","why","how",
    "into","over","under","about","between","among","during","before","after",
}

def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def tokenize_query(q: str) -> List[str]:
    q = q.lower()
    toks = TOKEN_RE.findall(q)
    clean = []
    for t in toks:
        if t in STOPWORDS:
            continue
        if len(t) < 2:
            continue
        clean.append(t)
    return clean

def compute_idf(df: int, N: int) -> float:
    # idf classic
    # avoid division by zero and idf weird negatives
    return math.log((N + 1) / (df + 1)) + 1.0

def tf_weight(tf: int) -> float:
    # log normalization
    return 1.0 + math.log(tf) if tf > 0 else 0.0

def rank_tfidf(
    query: str,
    postings: Dict[str, List[Tuple[str, int]]],
    df_map: Dict[str, int],
    N: int
) -> Dict[str, float]:
    """
    Score(d) = sum_{t in q} (tf_weight(tf_{t,d}) * idf(t))
    """
    q_terms = tokenize_query(query)
    if not q_terms:
        return {}

    scores = defaultdict(float)

    for t in q_terms:
        df = df_map.get(t, 0)
        if df == 0:
            continue
        idf = compute_idf(df, N)
        for doc_id, tf in postings.get(t, []):
            scores[doc_id] += tf_weight(tf) * idf

    return dict(scores)

def pretty_print(results: List[Tuple[str, float]], doc_meta: Dict[str, dict], k: int = 10) -> None:
    print(f"\nTop {min(k, len(results))} results:")
    for i, (did, score) in enumerate(results[:k], 1):
        m = doc_meta.get(did, {})
        title = m.get("title", "(no title)")
        year = m.get("year", "")
        venue = m.get("venue", "")
        url = m.get("url", "")
        print(f"{i:02d}. [{score:.4f}] {title}")
        print(f"    {year} | {venue}")
        if url:
            print(f"    {url}")

if __name__ == "__main__":
    postings = load_pickle(POSTINGS_PATH)
    df_map = load_pickle(VOCAB_PATH)
    doc_lengths = load_pickle(DOC_LEN_PATH)
    doc_meta = load_pickle(DOC_META_PATH)

    N = len(doc_lengths)
    print(f"Loaded index for TF-IDF ranking. N={N} documents.")

    print("\nRanked Retrieval (TF-IDF) ready. Type 'exit' to quit.\n")

    while True:
        q = input("query> ").strip()
        if not q:
            continue
        if q.lower() in {"exit", "quit"}:
            break

        scores = rank_tfidf(q, postings, df_map, N)
        if not scores:
            print("No results (query terms not in vocabulary).")
            continue

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        pretty_print(ranked, doc_meta, k=10)


Loaded index for TF-IDF ranking. N=613 documents.

Ranked Retrieval (TF-IDF) ready. Type 'exit' to quit.

query> document verification blockchain

Top 10 results:
01. [24.6180] Digital Document Verification using Blockchain
    2025 | International Journal for Research in Applied Science and Engineering Technology
    https://doi.org/10.22214/ijraset.2025.68159
02. [23.4185] A Blockchain Based Scalable Framework for Academic Document Verification
    2023 | None
    https://doi.org/10.1109/iccit60459.2023.10441245
03. [23.0080] Digital Document Verification System Using Blockchain
    2024 | International Journal For Multidisciplinary Research
    https://doi.org/10.36948/ijfmr.2024.v06i02.16798
04. [22.7112] Automated Medical Document Verification on Cloud Computing Platform: Blockchain-Based Soulbound Tokens
    2023 | Acta Informatica Pragensia
    https://doi.org/10.18267/j.aip.218
05. [21.7471] Blockchain-Based Decentralized Document Verification and Its Applications
    2025 | Jo

## 7. Evaluation

The search engine is evaluated using manually annotated relevance judgments.

Metrics:
- Precision@10
- Recall@10
- F1@10
- Average Precision (AP)
- Mean Average Precision (MAP)

Queries are aligned with the project motivation.


In [ ]:
import os
import re
import json
import math
import pickle
from collections import defaultdict
from typing import Dict, List, Tuple

POSTINGS_PATH = "data/index/postings.pkl"
VOCAB_PATH    = "data/index/vocab.pkl"
DOC_META_PATH = "data/index/doc_meta.pkl"

EVAL_DIR = "data/eval"
QRELS_PATH = os.path.join(EVAL_DIR, "qrels.json")

K = 10
RESET_QRELS = True

TOKEN_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

STOPWORDS = {
    "a","an","the","and","or","but","if","while","with","without","of","to","in","on","for","from",
    "by","as","at","is","are","was","were","be","been","being","this","that","these","those",
    "it","its","they","them","their","we","our","you","your","i","me","my",
    "can","could","should","would","may","might","must","will","shall",
    "not","no","nor","do","does","did","doing","done",
    "have","has","had","having",
    "also","such","than","then","there","here","when","where","why","how",
    "into","over","under","about","between","among","during","before","after",
}

# =========================
# QUERIES (aligned with motivation)
# - Document verification/authenticity
# - Trust + AI-generated text
# - Hallucinations / factuality / groundedness / claim verification
# - Misinformation, but connected to AI/LLM
# =========================
QUERIES = [
    "document verification",
    "document authenticity",
    "signature verification",

    "identity document verification",

    "claim verification",
    "evidence based verification",
    "fact checking claim verification",

    "llm hallucination detection",
    "factuality evaluation llm",
    "groundedness retrieval augmented generation",
    "rag hallucination detection",

    "ai generated misinformation detection",
]

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def tokenize_query(q: str) -> List[str]:
    q = q.lower()
    toks = TOKEN_RE.findall(q)
    return [t for t in toks if t not in STOPWORDS and len(t) >= 2]

def compute_idf(df: int, N: int) -> float:
    return math.log((N + 1) / (df + 1)) + 1.0

def tf_weight(tf: int) -> float:
    return 1.0 + math.log(tf) if tf > 0 else 0.0

def rank_tfidf(
    query: str,
    postings: Dict[str, List[Tuple[str, int]]],
    df_map: Dict[str, int],
    N: int
) -> List[Tuple[str, float]]:
    q_terms = tokenize_query(query)
    scores = defaultdict(float)

    for t in q_terms:
        df = df_map.get(t, 0)
        if df == 0:
            continue
        idf = compute_idf(df, N)
        for doc_id, tf in postings.get(t, []):
            scores[doc_id] += tf_weight(tf) * idf

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def precision_at_k(rels: List[int], k: int) -> float:
    rels = rels[:k]
    return sum(rels) / k if k > 0 else 0.0

def recall_at_k(rels: List[int], total_relevant: int, k: int) -> float:
    if total_relevant == 0:
        return 0.0
    return sum(rels[:k]) / total_relevant

def f1(p: float, r: float) -> float:
    return (2*p*r/(p+r)) if (p+r) > 0 else 0.0

def average_precision_at_k(rels: List[int], k: int) -> float:
    rels = rels[:k]
    num_rel = 0
    ap_sum = 0.0
    for i, r in enumerate(rels, 1):
        if r == 1:
            num_rel += 1
            ap_sum += num_rel / i
    return ap_sum / num_rel if num_rel > 0 else 0.0

def load_qrels() -> Dict:
    if not os.path.exists(QRELS_PATH):
        return {}
    with open(QRELS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_qrels(qrels: Dict) -> None:
    with open(QRELS_PATH, "w", encoding="utf-8") as f:
        json.dump(qrels, f, ensure_ascii=False, indent=2)

def show_topk(query: str, ranked: List[Tuple[str, float]], doc_meta: Dict[str, dict], k: int) -> List[str]:
    print("\n" + "="*80)
    print(f"QUERY: {query}")
    print("="*80)
    top_ids = []
    for i, (did, score) in enumerate(ranked[:k], 1):
        m = doc_meta.get(did, {})
        title = m.get("title", "(no title)")
        year = m.get("year", "")
        venue = m.get("venue", "")
        print(f"{i:02d}. [{score:.4f}] {did}")
        print(f"    {title}")
        print(f"    {year} | {venue}")
        top_ids.append(did)
    return top_ids

if __name__ == "__main__":
    ensure_dir(EVAL_DIR)

    postings = load_pickle(POSTINGS_PATH)
    df_map = load_pickle(VOCAB_PATH)
    doc_meta = load_pickle(DOC_META_PATH)
    N = len(doc_meta)

    if RESET_QRELS and os.path.exists(QRELS_PATH):
        os.remove(QRELS_PATH)

    qrels = load_qrels()

    print("Evaluation setup:")
    print(f"- Queries: {len(QUERIES)}")
    print(f"- k: {K}")
    print(f"- qrels file: {QRELS_PATH}")
    print(f"- reset: {RESET_QRELS}\n")

    # 1) Relevance judgments
    for q in QUERIES:
        if q in qrels:
            continue

        ranked = rank_tfidf(q, postings, df_map, N)
        top_ids = show_topk(q, ranked, doc_meta, k=K)

        print("\nMark relevance for TOP-k results.")
        print("Type a list of ranks that are relevant, e.g.: 1 2 5")
        print("Press Enter for none relevant.\n")

        s = input("relevant ranks> ").strip()
        rel = [0]*len(top_ids)
        if s:
            ranks = set(int(x) for x in s.split() if x.isdigit())
            for i in range(1, len(top_ids)+1):
                if i in ranks:
                    rel[i-1] = 1

        qrels[q] = {"doc_ids": top_ids, "relevance": rel}
        save_qrels(qrels)
        print("Saved qrels.\n")

    # 2) Metrics
    print("\n" + "#"*80)
    print(f"METRICS (at k={K})")
    print("#"*80)

    p_list, r_list, f_list, ap_list = [], [], [], []

    for q in QUERIES:
        rels = qrels[q]["relevance"]
        total_rel = sum(rels)

        p = precision_at_k(rels, K)
        r = recall_at_k(rels, total_rel, K)
        f = f1(p, r)
        ap = average_precision_at_k(rels, K)

        p_list.append(p)
        r_list.append(r)
        f_list.append(f)
        ap_list.append(ap)

        print(f"\nQuery: {q}")
        print(f"  Relevant in top-{K}: {total_rel}/{K}")
        print(f"  Precision@{K}: {p:.3f}")
        print(f"  Recall@{K}:    {r:.3f}")
        print(f"  F1@{K}:        {f:.3f}")
        print(f"  AP@{K}:        {ap:.3f}")

    print("\n" + "-"*80)
    print(f"Mean Precision@{K}: {sum(p_list)/len(p_list):.3f}")
    print(f"Mean Recall@{K}:    {sum(r_list)/len(r_list):.3f}")
    print(f"Mean F1@{K}:        {sum(f_list)/len(f_list):.3f}")
    print(f"MAP@{K}:            {sum(ap_list)/len(ap_list):.3f}")
    print("-"*80)


Evaluation setup:
- Queries: 12
- k: 10
- qrels file: data/eval/qrels.json
- reset: True


QUERY: document verification
01. [18.1733] https://openalex.org/W4403277433
    Transformer-Based Approach for Automatic Semantic Financial Document Verification
    2024 | IEEE Access
02. [15.3576] https://openalex.org/W4409539148
    Digital Document Verification using Blockchain
    2025 | International Journal for Research in Applied Science and Engineering Technology
03. [14.1308] https://openalex.org/W4307923080
    A Framework for Online Document Verification Using Self-Sovereign Identity Technology
    2022 | Sensors
04. [13.7476] https://openalex.org/W4394796098
    Digital Document Verification System Using Blockchain
    2024 | International Journal For Multidisciplinary Research
05. [13.6985] https://openalex.org/W4385963833
    Document Verification using Blockchain
    2023 | Zenodo (CERN European Organization for Nuclear Research)
06. [13.6531] https://openalex.org/W2526177595
    

## 8. Results and Discussion

The system achieves strong performance, particularly for queries related to:

- Document verification
- Identity verification
- LLM hallucination detection
- RAG-based hallucination mitigation

The Mean Average Precision (MAP) indicates that TF-IDF is effective
for retrieving relevant academic literature in this domain.

Some queries (e.g. claim verification) show lower precision due to
topic overlap with broader IR and NLP literature.


## 9.User Interface

A simple command-line interface (CLI) is implemented directly in the Google Colab notebook using ipywidgets.
Users can input queries and view the retrieved results in the console.

The interface supports:

* Boolean queries using logical operators (AND, OR, NOT)
* Ranked retrieval queries using TF-IDF

Results are displayed with basic metadata (title, year, venue, URL), and TF-IDF scores for ranked retrieval.


In [ ]:
# Notebook widget support (VS Code, Jupyter and Google Colab).
# Install requirements.txt in the selected Python kernel if widgets are missing.
import importlib.util
if importlib.util.find_spec("ipywidgets") is None:
    raise ImportError("Install ipywidgets in this kernel: %pip install ipywidgets")
if importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None:
    from google.colab import output
    output.enable_custom_widget_manager()


In [ ]:
import re
import math
import pickle
from collections import defaultdict
from typing import Dict, List, Tuple, Set
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# -------------------------
# Paths
# -------------------------
POSTINGS_PATH = "data/index/postings.pkl"
VOCAB_PATH    = "data/index/vocab.pkl"
DOC_LEN_PATH  = "data/index/doc_lengths.pkl"
DOC_META_PATH = "data/index/doc_meta.pkl"

# Locate generated data whether the kernel starts in the repo or notebook folder.
from pathlib import Path
if not Path(POSTINGS_PATH).is_file() and Path("notebooks/data/index/postings.pkl").is_file():
    POSTINGS_PATH = "notebooks/" + POSTINGS_PATH
    VOCAB_PATH = "notebooks/" + VOCAB_PATH
    DOC_LEN_PATH = "notebooks/" + DOC_LEN_PATH
    DOC_META_PATH = "notebooks/" + DOC_META_PATH

# -------------------------
# Helpers: load
# -------------------------
def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

postings = load_pickle(POSTINGS_PATH)   # term -> list[(doc_id, tf)]
df_map   = load_pickle(VOCAB_PATH)      # term -> df
doc_len  = load_pickle(DOC_LEN_PATH)    # doc_id -> len_tokens
doc_meta = load_pickle(DOC_META_PATH)   # doc_id -> metadata
universe = set(doc_len.keys())
N = len(doc_len)

TOKEN_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

STOPWORDS = {
    "a","an","the","and","or","but","if","while","with","without","of","to","in","on","for","from",
    "by","as","at","is","are","was","were","be","been","being","this","that","these","those",
    "it","its","they","them","their","we","our","you","your","i","me","my",
    "can","could","should","would","may","might","must","will","shall",
    "not","no","nor","do","does","did","doing","done",
    "have","has","had","having",
    "also","such","than","then","there","here","when","where","why","how",
    "into","over","under","about","between","among","during","before","after",
}

# =========================================================
# Boolean retrieval
# =========================================================
PRECEDENCE = {"NOT": 3, "AND": 2, "OR": 1}
RIGHT_ASSOC = {"NOT"}

def normalize_query_tokens(q: str) -> List[str]:
    q = q.strip().replace("(", " ( ").replace(")", " ) ")
    parts = q.split()
    out = []
    for p in parts:
        up = p.upper()
        if up in {"AND", "OR", "NOT", "(", ")"}:
            out.append(up)
        else:
            out.extend(TOKEN_RE.findall(p.lower()))
    return out

def get_docs_for_term(term: str) -> Set[str]:
    return set(doc_id for doc_id, _tf in postings.get(term, []))

def to_rpn(tokens: List[str]) -> List[str]:
    output, ops = [], []
    for t in tokens:
        if t in {"AND","OR","NOT"}:
            while ops:
                top = ops[-1]
                if top == "(":
                    break
                if (PRECEDENCE[top] > PRECEDENCE[t]) or (PRECEDENCE[top] == PRECEDENCE[t] and t not in RIGHT_ASSOC):
                    output.append(ops.pop())
                else:
                    break
            ops.append(t)
        elif t == "(":
            ops.append(t)
        elif t == ")":
            while ops and ops[-1] != "(":
                output.append(ops.pop())
            if not ops:
                raise ValueError("Mismatched parentheses.")
            ops.pop()
        else:
            output.append(t)

    while ops:
        top = ops.pop()
        if top in {"(", ")"}:
            raise ValueError("Mismatched parentheses.")
        output.append(top)

    return output

def eval_rpn(rpn: List[str]) -> Set[str]:
    stack: List[Set[str]] = []
    for t in rpn:
        if t == "NOT":
            if not stack:
                raise ValueError("NOT missing operand.")
            a = stack.pop()
            stack.append(universe - a)
        elif t in {"AND","OR"}:
            if len(stack) < 2:
                raise ValueError(f"{t} missing operand(s).")
            b = stack.pop()
            a = stack.pop()
            stack.append(a & b if t == "AND" else a | b)
        else:
            stack.append(get_docs_for_term(t))

    if len(stack) != 1:
        raise ValueError("Invalid boolean query.")
    return stack[0]

# =========================================================
# TF-IDF ranked retrieval
# =========================================================
def tokenize_query(q: str) -> List[str]:
    toks = TOKEN_RE.findall(q.lower())
    return [t for t in toks if t not in STOPWORDS and len(t) >= 2]

def compute_idf(df: int, N: int) -> float:
    return math.log((N + 1) / (df + 1)) + 1.0

def tf_weight(tf: int) -> float:
    return 1.0 + math.log(tf) if tf > 0 else 0.0

def rank_tfidf(query: str) -> Dict[str, float]:
    q_terms = tokenize_query(query)
    if not q_terms:
        return {}
    scores = defaultdict(float)
    for t in q_terms:
        df = df_map.get(t, 0)
        if df == 0:
            continue
        idf = compute_idf(df, N)
        for doc_id, tf in postings.get(t, []):
            scores[doc_id] += tf_weight(tf) * idf
    return dict(scores)

# =========================================================
# Pretty HTML results
# =========================================================
def render_results_table(rows: List[dict], title: str):
    if not rows:
        display(HTML(f"<b>{title}</b><br>No results."))
        return

    html = [f"<h4 style='margin:10px 0 6px 0'>{title}</h4>"]
    html.append("<table style='border-collapse:collapse; width:100%'>")
    html.append("<tr>"
                "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px'>#</th>"
                "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px'>Title</th>"
                "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px'>Year</th>"
                "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px'>Venue</th>"
                "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px'>Link</th>"
                "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px'>Score</th>"
                "</tr>")

    for i, r in enumerate(rows, 1):
        t = (r.get("title") or "(no title)").replace("<","&lt;").replace(">","&gt;")
        y = r.get("year") or ""
        v = (r.get("venue") or "")
        url = r.get("url") or ""
        score = r.get("score")
        score_str = f"{score:.4f}" if isinstance(score, (int,float)) else ""

        link_html = f"<a href='{url}' target='_blank'>open</a>" if url else ""
        html.append("<tr>"
                    f"<td style='padding:6px; vertical-align:top'>{i}</td>"
                    f"<td style='padding:6px; vertical-align:top'>{t}</td>"
                    f"<td style='padding:6px; vertical-align:top'>{y}</td>"
                    f"<td style='padding:6px; vertical-align:top'>{v}</td>"
                    f"<td style='padding:6px; vertical-align:top'>{link_html}</td>"
                    f"<td style='padding:6px; vertical-align:top'>{score_str}</td>"
                    "</tr>")
    html.append("</table>")
    display(HTML("".join(html)))

# =========================================================
# UI Widgets
# =========================================================
mode = widgets.ToggleButtons(
    options=["TF-IDF (Ranked)", "Boolean (Filter)"],
    value="TF-IDF (Ranked)",
    description="Mode:",
    style={"description_width": "initial"}
)

query_box = widgets.Text(
    value="document verification blockchain",
    description="Query:",
    placeholder="Type your query...",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%")
)

topk = widgets.IntSlider(
    value=10, min=5, max=30, step=1,
    description="Top-k:",
    style={"description_width": "initial"}
)

run_btn = widgets.Button(description="Search", button_style="primary")
out = widgets.Output()

def run_search(_):
    with out:
        clear_output()
        q = query_box.value.strip()
        if not q:
            display(HTML("<b>Please type a query.</b>"))
            return

        try:
            if mode.value.startswith("TF-IDF"):
                scores = rank_tfidf(q)
                if not scores:
                    display(HTML("<b>No results (query terms not in vocabulary).</b>"))
                    return

                ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk.value]
                rows = []
                for doc_id, sc in ranked:
                    m = doc_meta.get(doc_id, {})
                    rows.append({
                        "title": m.get("title"),
                        "year": m.get("year"),
                        "venue": m.get("venue"),
                        "url": m.get("url"),
                        "score": sc
                    })
                render_results_table(rows, title=f"TF-IDF results for: {q}")

            else:
                tokens = normalize_query_tokens(q)
                rpn = to_rpn(tokens)
                result_set = eval_rpn(rpn)

                # order like you already do: year desc then citations desc
                result_list = list(result_set)
                result_list.sort(
                    key=lambda did: (
                        doc_meta.get(did, {}).get("year") or 0,
                        doc_meta.get(did, {}).get("citationCount") or 0
                    ),
                    reverse=True
                )
                result_list = result_list[:topk.value]

                rows = []
                for doc_id in result_list:
                    m = doc_meta.get(doc_id, {})
                    rows.append({
                        "title": m.get("title"),
                        "year": m.get("year"),
                        "venue": m.get("venue"),
                        "url": m.get("url"),
                        "score": None
                    })
                render_results_table(rows, title=f"Boolean results for: {q}  (showing top {topk.value})")

        except Exception as e:
            display(HTML(f"<b>Error:</b> {str(e)}"))

run_btn.on_click(run_search)

ui = widgets.VBox([
    widgets.HTML("<h3>Search Engine UI</h3><p>Type a query, choose a mode, and press Search.</p>"),
    widgets.HBox([mode, topk]),
    widgets.HBox([query_box, run_btn]),
    out
])

display(ui)


## 9. Conclusions

This project demonstrates that classical Information Retrieval techniques
remain effective for exploring modern research topics such as:

- AI-generated misinformation
- Trustworthiness of LLM outputs
- Document and claim verification

Future work could include:
- BM25 ranking
- Neural re-ranking
- Query expansion
- Integration with RAG pipelines
